# Phase 4: Individual Baseline Models

This notebook presents fixed-parameter Ridge, linear SVM, and XGBoost results for each reproduced feature view. Models are fitted on training data only. No hyperparameter optimization, threshold tuning, or MulCas cascade is performed.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

results_path = PROJECT_ROOT / "results" / "baseline" / "individual_baseline_results.json"
if not results_path.exists():
    raise FileNotFoundError(
        "Run '.\\.venv\\Scripts\\python.exe -m src.models.run_baselines' first."
    )

results = json.loads(results_path.read_text(encoding="utf-8"))
records = pd.DataFrame(results["records"])
results["selection_policy"]

## Fixed Baseline Parameters

In [ ]:
parameter_table = records[["model", "parameters", "seed"]].drop_duplicates("model")
parameter_table.reset_index(drop=True)

## Validation Results

Validation results may inform later model-selection phases. They do not alter the fixed Phase 4 parameters.

In [ ]:
metric_columns = ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "pr_auc", "roc_auc"]
validation = records[records["split"] == "validation"]
validation[["model", "feature_view", *metric_columns]].sort_values("f1", ascending=False).reset_index(drop=True)

In [ ]:
validation_f1 = validation.pivot(index="feature_view", columns="model", values="f1")
ax = validation_f1[["ridge", "svm", "xgboost"]].plot.bar()
ax.set_title("Validation F1 by feature view and model")
ax.set_xlabel("Feature view")
ax.set_ylabel("F1")
ax.set_ylim(0, 1)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()

## Final-Test Reporting

The final-test partition is reported only after fitting fixed models. Its results are not used to change parameters or decision thresholds.

In [ ]:
test = records[records["split"] == "test"]
test[["model", "feature_view", *metric_columns]].sort_values("f1", ascending=False).reset_index(drop=True)

In [ ]:
test_pr_auc = test.pivot(index="feature_view", columns="model", values="pr_auc")
ax = test_pr_auc[["ridge", "svm", "xgboost"]].plot.bar()
ax.set_title("Final-test PR-AUC by feature view and model")
ax.set_xlabel("Feature view")
ax.set_ylabel("PR-AUC")
ax.set_ylim(0, 1)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()

## Confusion-Matrix Counts

In [ ]:
records[["split", "model", "feature_view", "tn", "fp", "fn", "tp"]].sort_values(
    ["split", "feature_view", "model"]
).reset_index(drop=True)